# LSI‑lite (Colab) — Figure / Mark‑Making / Landscape

Small, profiled composition gate for quick QA of images.  
**Signals:** Δx (centroid offset), rᵥ (void ratio / fill), ρᵣ (Laplacian stroke energy).  
**Profiles:** `Figure_Default`, `MarkMaking_Expressive`, `Landscape` (band guards + weights).  
**Output:** per‑image table + plots + CSV/HTML; acceptance = (LSI_lite_100 ≥ gate) & no RED bands.

### Quick start
1. Run **Setup & Config**. (Pins are optional and off by default to avoid long installs.)
2. Run **Reset** (clears `/content/images`).
3. Use **Option A** to upload images (preferred). ZIP/Drive are fallbacks.
4. Pick a **profile mode** (keep `Auto` for smoke tests; fix to a preset for formal runs).
5. Run **Run scoring**, then **Plots**, then **Export**.

> This is **not** recognition or a “style police”. It’s a tiny, defensible ruler over three primitives.


In [ ]:
# @title Setup & Config (resilient, pinned when requested)
PIN_DEPS = True  # @param {type:"boolean"}

PINS = {
    "opencv-python-headless": "4.10.0.84",
    "numpy": "1.26.4",
    "pandas": "2.0.3",
    "matplotlib": "3.7.5",
    "Pillow": "10.4.0",
}

def maybe_pin_deps(pin=PIN_DEPS):
    """Install pinned versions only if requested; otherwise use Colab defaults."""
    if not pin:
        print("Pinned deps disabled. Using environment defaults.")
        return
    import sys, subprocess
    pkgs = [f"{k}=={v}" for k, v in PINS.items()]
    cmd = [sys.executable, "-m", "pip", "install",
           "--prefer-binary", "--only-if-needed", "--upgrade-strategy", "only-if-needed"] + pkgs
    print("Installing pinned:", " ".join(pkgs))
    try:
        subprocess.check_call(cmd)
        print("Pin install finished.")
    except Exception as e:
        print("Pin install failed; continuing with existing env:", e)

maybe_pin_deps()

# now the normal imports
import os, glob, math, io, json, shutil, base64
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

os.makedirs("/content/images", exist_ok=True)

CONFIG = {
    "preprocessing": {"longest_side": 1536, "morph_kernel": 5},
    "accept": {"gate_100": 55.0},
    "sigma_scale": 0.35,
}

PROFILES = {
    "Figure_Default": {
        "weights": {"dx": 0.45, "rv": 0.35, "rho": 0.20},
        "bands": {
            "dx":  {"guard": [0.05, 0.85]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.10, 0.80]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.08},
        "dx_roi": None,
    },
    "MarkMaking_Expressive": {
        "weights": {"dx": 0.40, "rv": 0.30, "rho": 0.30},
        "bands": {
            "dx":  {"guard": [0.02, 0.90]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.06, 0.70]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.12},  # ← keep only this one
        "dx_roi": None,
    },
    "Landscape": {
        "weights": {"dx": 0.35, "rv": 0.40, "rho": 0.25},
        "bands": {
            "dx":  {"guard": [0.05, 0.95]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.08, 0.80]},
        },
        "rho_mask": {"type": "full"},
        "dx_roi": None,   # ← turn on reflection-aware Δx crop (or set to None to disable)
    },
}
print("Ready. Profiles:", list(PROFILES.keys()))

In [ ]:
# @title Helpers (robust loader, safe largest, morphology, masks, auto-profile)

def load_image_robust(path_or_bytes):
    """Open as RGB with EXIF orientation; return grayscale uint8 and original RGB HxWx3."""
    if isinstance(path_or_bytes, (bytes, bytearray)):
        img = Image.open(io.BytesIO(path_or_bytes))
    else:
        img = Image.open(path_or_bytes)
    img = ImageOps.exif_transpose(img).convert("RGB")
    rgb = np.asarray(img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return gray, rgb

def resize_longest(gray, longest):
    h, w = gray.shape[:2]
    s = longest / max(h, w)
    if s < 1.0:
        gray = cv2.resize(gray, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
    return gray

def safe_largest(bin_u8: np.ndarray):
    cnts, _ = cv2.findContours(bin_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    a = max(cnts, key=cv2.contourArea)
    mask = np.zeros_like(bin_u8, dtype=np.uint8)
    cv2.drawContours(mask, [a], -1, 255, -1)
    return mask

def _morph(mask, k):
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    return mask

def foreground_mask(gray: np.ndarray, cfg: dict) -> np.ndarray:
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, bin_ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Prefer “dark subject on light paper”, then try normal, then degenerate fallback
    mask = safe_largest((255 - bin_).astype(np.uint8))
    if mask is None:
        mask = safe_largest(bin_.astype(np.uint8))
    if mask is None:
        mask = (bin_ > 0).astype(np.uint8) * 255

    # Light clean-up
    k = cfg["preprocessing"]["morph_kernel"]
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, 1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, 1)
    return mask

def subject_halo_mask(fg_mask, halo_frac=0.10):
    h, w = fg_mask.shape[:2]
    r = max(1, int(halo_frac * min(h, w)))
    kernel = np.ones((r, r), np.uint8)
    halo = cv2.dilate(fg_mask, kernel, iterations=1)
    return halo

def auto_profile(dx, rv):
    # very small dx + high rv => Landscape; mid dx / mid rv => Figure; else MarkMaking
    if rv >= 0.35 and dx <= 0.25:
        return "Landscape"
    if rv <= 0.55 and dx >= 0.15:
        return "Figure_Default"
    return "MarkMaking_Expressive"

def band_center_and_span(profile: str, band: str):
    """Return (center, half-span) from the guard band for a given profile/band."""
    g = PROFILES[profile]["bands"][band]["guard"]  # [lo, hi]
    c = 0.5 * (g[0] + g[1])
    s = max(1e-6, 0.5 * (g[1] - g[0]))
    return c, s

def ensure_gray_array(x):
    """Accept path | ndarray | tuple and return a 2-D uint8 grayscale array."""
    # If it's a path/bytes, load it
    if isinstance(x, (str, bytes, bytearray)):
        g = load_image_robust(x)
    else:
        g = x

    # Some code paths hand us (gray, extra). Take the first item.
    if isinstance(g, tuple):
        g = g[0]

    g = np.asarray(g)
    # If RGB/BGR, convert to gray
    if g.ndim == 3 and g.shape[2] in (3, 4):
        # assume RGB because load_image_robust returns RGB
        g = cv2.cvtColor(g, cv2.COLOR_RGB2GRAY)
    # Ensure uint8
    if g.dtype != np.uint8:
        g = np.clip(g, 0, 255).astype(np.uint8)
    return g

def auto_dx_roi(gray_or_tuple,
                *,
                top_frac: float = 0.60,
                reflect_thr: float = 0.35,     # was 0.50; looser to catch real lakes
                waterish_ratio: float = 1.15,  # was 1.30; horizontal > vertical
                require_symmetry: bool = False,
                sym_eps: float = 0.25):
    """
    Auto-crop ROI for Δx when a water reflection is present (Landscape).
    Accepts path/ndarray/(gray,rgb) tuple. Returns dict {"type":"top_frac","top":<f>}
    or None when no crop is recommended.

    Heuristic:
      1) Reflection similarity between top half and flipped bottom half (edges).
      2) "Water-ish" bottom: horizontal > vertical gradients.
      3) (optional) Similar foreground fill in top/bottom halves.
    """
    # 1) Get a 2-D uint8 grayscale image, regardless of input type
    g = ensure_gray_array(gray_or_tuple)   # your helper: path/tuple-safe → 2-D uint8
    if g is None or g.ndim != 2:
        return None
    H, W = g.shape
    if H < 4 or W < 4:
        return None

    h2  = H // 2
    top = g[:h2, :]
    bot = g[h2:, :]

    # 2) Edge maps for similarity probe
    e_top = cv2.Canny(top, 50, 100).astype(np.float32)
    e_bot = cv2.Canny(bot, 50, 100).astype(np.float32)
    e_bot_flip = np.flipud(e_bot)

    # Cosine similarity between the two halves’ edge maps
    num = (e_top * e_bot_flip).sum()
    den = float(np.linalg.norm(e_top) * np.linalg.norm(e_bot_flip) + 1e-6)
    reflect_sim = num / den

    # 3) "Water-ish" check: horizontal structure should dominate in the bottom half
    gx = np.abs(cv2.Sobel(bot, cv2.CV_32F, 1, 0, ksize=3)).mean()
    gy = np.abs(cv2.Sobel(bot, cv2.CV_32F, 0, 1, ksize=3)).mean()
    waterish = gx > (waterish_ratio * gy)

    # 4) Optional symmetric foreground fill (uses your existing CONFIG + mask helper)
    ok_symmetry = True
    if require_symmetry:
        mask = foreground_mask(g, CONFIG)       # relies on global CONFIG (as in your notebook)
        fill_top = (mask[:h2, :] > 0).mean()
        fill_bot = (mask[h2:, :] > 0).mean()
        ok_symmetry = abs(fill_top - fill_bot) < sym_eps

    # 5) Decision
    if (reflect_sim > reflect_thr) and waterish and ok_symmetry:
        return {"type": "top_frac", "top": float(top_frac)}
    return None

In [ ]:

# @title Core primitives

def centroid_delta_x(gray, cfg, dx_roi=None, mode="hybrid_masked"):
    g = ensure_gray_array(gray)

    # Optional Landscape crop (top fraction)
    if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
        top = float(dx_roi.get("top", 0.60))
        H = g.shape[0]
        g = g[: max(1, int(H * top)), :]

    g = resize_longest(g, cfg["preprocessing"]["longest_side"])
    g = cv2.bilateralFilter(g, d=5, sigmaColor=25, sigmaSpace=25)

    # Foreground mask
    fg = foreground_mask(g, cfg)
    fg_u8 = (fg > 0).astype(np.uint8) * 255

    # Edge map
    e = cv2.Canny(g, 50, 100).astype(np.uint8)

    # --- masked-edges first (hybrid), then fallback to mask centroid ---
    m = None
    if mode in ("edge", "edge_masked", "hybrid_masked"):
        e_use = e if mode == "edge" else cv2.bitwise_and(e, fg_u8)
        m = cv2.moments(e_use)

    if m is None or m["m00"] <= 1e-6:
        m = cv2.moments(fg_u8)
        if m["m00"] <= 1e-6:
            return 0.0

    cx = m["m10"] / (m["m00"] + 1e-6)
    W  = g.shape[1]
    dx = abs(cx - (W / 2.0)) / (W / 2.0)
    return float(np.clip(dx, 0.0, 1.0))

def void_ratio(gray, cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    mask = foreground_mask(gray, cfg)
    fill = (mask > 0).mean()
    rv = float(np.clip(1.0 - fill, 0.0, 1.0))  # more background = higher void
    return rv

def rho_r(gray, cfg, rho_mask_cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    gray = cv2.bilateralFilter(gray, d=5, sigmaColor=25, sigmaSpace=25)
    fg = foreground_mask(gray, cfg)
    if rho_mask_cfg.get("type") == "subject_halo":
        m = subject_halo_mask(fg, halo_frac=rho_mask_cfg.get("halo_frac", 0.10))
    else:
        m = np.ones_like(fg, dtype=np.uint8)*255
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    energy = np.abs(lap) / 255.0
    sel = energy[m > 0]
    if sel.size == 0:
        return 0.0
    val = float(sel.mean())
    # gentle scale to [0,1]
    return float(np.clip(val * CONFIG.get("rho_scale", 5.0), 0.0, 1.0))

def measure_primitives(path_or_gray, cfg, rho_mask_cfg, dx_roi=None):
    gray = load_image_robust(path_or_gray) if isinstance(path_or_gray, (str, bytes)) else path_or_gray
    dx  = centroid_delta_x(gray, cfg, dx_roi=dx_roi)
    rv  = void_ratio(gray, cfg)
    rho = rho_r(gray, cfg, rho_mask_cfg)
    return {"dx": dx, "rv": rv, "rho": rho}


In [ ]:

# @title Scoring kernel

def band_flag(val, guard):
    lo, hi = guard
    return "OK" if (lo <= val <= hi) else "RED"

def gaussian_score(val, guard):
    lo, hi = guard
    c = 0.5 * (lo + hi)
    sigma = max(1e-6, CONFIG.get("sigma_scale", 0.25) * (hi - lo))
    s = math.exp(-0.5 * ((val - c) / sigma) ** 2)
    return float(np.clip(s, 0.0, 1.0))

def score_with_profile(prims, profile_cfg, cfg):
    bands = profile_cfg["bands"]
    w = profile_cfg["weights"]
    # band flags
    b_dx  = band_flag(prims["dx"],  bands["dx"]["guard"])
    b_rv  = band_flag(prims["rv"],  bands["rv"]["guard"])
    b_rho = band_flag(prims["rho"], bands["rho"]["guard"])

    # soft band scores
    s_dx  = gaussian_score(prims["dx"],  bands["dx"]["guard"])
    s_rv  = gaussian_score(prims["rv"],  bands["rv"]["guard"])
    s_rho = gaussian_score(prims["rho"], bands["rho"]["guard"])

    # weighted geometric mean (epsilon-safe)
    eps = 1e-6
    k = (
        max(eps, s_dx )**w["dx"] *
        max(eps, s_rv )**w["rv"] *
        max(eps, s_rho)**w["rho"]
    ) ** (1.0 / (w["dx"] + w["rv"] + w["rho"]))

    LSI = 100.0 * k
    accepted = (LSI >= cfg["accept"]["gate_100"]) and (b_dx!="RED") and (b_rv!="RED") and (b_rho!="RED")

    return {
        "delta_x": round(prims["dx"], 3),
        "void_ratio": round(prims["rv"], 3),
        "rupture_rho": round(prims["rho"], 3),
        "K_lite": round(k, 3),
        "LSI_lite_100": round(LSI, 1),
        "band_delta_x": b_dx,
        "band_r_v": b_rv,
        "band_rho_r": b_rho,
        "accepted": bool(accepted),
    }


In [ ]:

# @title Reset: clear /content/images
import shutil, os
IMG_DIR = "/content/images"
if os.path.exists(IMG_DIR):
    shutil.rmtree(IMG_DIR)
os.makedirs(IMG_DIR, exist_ok=True)
print("Reset:", IMG_DIR)


In [ ]:

# @title Option A — Classic multiple‑file uploader (preferred)
from google.colab import files
uploaded = files.upload()
upload_paths = []
for name, data in uploaded.items():
    path = f"/content/images/{name}"
    with open(path, "wb") as f:
        f.write(data)
    upload_paths.append(path)
print("Uploaded:", len(upload_paths), "files")


In [ ]:

# @title Option B — Upload a ZIP of images (fallback)
from google.colab import files
import zipfile, io, os, glob
z = files.upload()
assert len(z)==1, "Upload exactly one .zip"
name, bytes_ = next(iter(z.items()))
assert name.lower().endswith(".zip"), "This path expects a .zip file"
with zipfile.ZipFile(io.BytesIO(bytes_), 'r') as zip_ref:
    zip_ref.extractall("/content/images")
upload_paths = sorted([p for p in glob.glob("/content/images/**/*", recursive=True)
                       if os.path.splitext(p)[1].lower() in [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]])
print("Extracted:", len(upload_paths), "images to /content/images")


In [ ]:

# @title Option C — Google Drive folder copy (batch)
from google.colab import drive
import shutil, os, glob
drive.mount('/content/drive')
DRIVE_FOLDER = ""  # @param {type:"string"}
assert DRIVE_FOLDER, "Set DRIVE_FOLDER to a Drive path, e.g. /content/drive/MyDrive/my_images"
ex = [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]
srcs = [p for p in glob.glob(os.path.join(DRIVE_FOLDER, "**/*"), recursive=True)
        if os.path.splitext(p)[1].lower() in ex]
for p in srcs:
    shutil.copy2(p, "/content/images/")
print("Copied:", len(srcs), "images into /content/images")


In [ ]:

# @title Profile selector (widget + fallback)
try:
    import ipywidgets as widgets
    profile_mode_widget = widgets.Dropdown(
        options=["Auto","Figure_Default","MarkMaking_Expressive","Landscape"],
        value="Auto",
        description="profile_mode",
    )
    display(profile_mode_widget)
except Exception as e:
    print("Widget not available; using string fallback. Set profile_mode manually.")
profile_mode = "MarkMaking_Expressive"  # @param ["Auto","Figure_Default","MarkMaking_Expressive","Landscape"]


In [ ]:
import matplotlib.image as mpimg
import glob, os

paths = sorted(glob.glob("/content/images/*"))
n = min(12, len(paths))
if n == 0:
    print("No images in /content/images yet. Use Option A2/A3 or B above.")
else:
    cols = 4; rows = (n + cols - 1)//cols
    plt.figure(figsize=(cols*3, rows*3))
    for i,p in enumerate(paths[:n], 1):
        plt.subplot(rows, cols, i)
        plt.imshow(mpimg.imread(p))
        plt.title(os.path.basename(p)[:30], fontsize=8); plt.axis("off")
    plt.show()

In [ ]:
# @title Run scoring (per-image loop)
# === Run scoring (per-image loop) ===
from glob import glob
import os, numpy as np, pandas as pd

# ---- 0) Gather inputs (whatever Option A/B/C uploaded) ----
exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff")
upload_paths = sorted([p for p in glob("/content/images/*")
                       if os.path.splitext(p)[1].lower() in exts])
print("Found", len(upload_paths), "images")

# ---- 1) Read chosen mode from the dropdown (or fallback string) ----
try:
    chosen_mode = profile_mode_widget.value
except NameError:
    chosen_mode = profile_mode
print("Using profile_mode:", chosen_mode)

rows = []
for path in upload_paths:
    # ---- load image once (EXIF-aware, robust); ignore RGB in this loop ----
    gray, _ = load_image_robust(path)

    # ---- choose profile ----
    if chosen_mode == "Auto":
        # quick, ROI-free probes just to decide a profile
        quick_dx = centroid_delta_x(gray, CONFIG, dx_roi=None)
        quick_rv = void_ratio(gray, CONFIG)
        pf = auto_profile(quick_dx, quick_rv)
    else:
        pf = chosen_mode

    # mask settings for this profile
    rho_mask_cfg = PROFILES[pf]["rho_mask"]

    # ---- optional ROI for Δx (Landscape only, if configured) ----
    dx_roi = None
    if pf == "Landscape" and PROFILES[pf].get("dx_roi") == "auto":
        dx_roi = auto_dx_roi(gray) or None     # returns {"type":"top_frac","top":0.60} or None

    # ---- compute ALL primitives ONCE (threads dx_roi into Δx) ----
    prims = measure_primitives(gray, CONFIG, rho_mask_cfg, dx_roi=dx_roi)

    # ---- score with the selected profile ----
    row = score_with_profile(prims, PROFILES[pf], CONFIG)
    row.update({"frame": os.path.basename(path), "profile": pf})
    rows.append(row)

# ---- build the base table ----
df = pd.DataFrame(rows).round(3)

# ----------------------------------------------------------------------------------
# Post-processing helper: band centers, distances, normalized offsets, and priority
# ----------------------------------------------------------------------------------

def band_center_and_span(profile_name: str, key: str):
    """
    Returns (center, span) for the guard band of 'key' in the given profile.
    key ∈ {"dx","rv","rho"}; center = midpoint; span = half-width.
    """
    lo, hi = PROFILES[profile_name]["bands"][key]["guard"]
    center = (lo + hi) / 2.0
    span   = (hi - lo) / 2.0
    return center, span

def add_band_distance_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()

    # Column names tolerant to either schema
    dxcol  = "delta_x"     if "delta_x"     in out.columns else "dx"
    rvcol  = "void_ratio"  if "void_ratio"  in out.columns else "rv"
    rhocol = "rupture_rho" if "rupture_rho" in out.columns else "rho"

    # Per-row band centers & spans from the row's profile
    centers_dx  = out["profile"].apply(lambda pf: band_center_and_span(pf, "dx")[0]).astype(float)
    centers_rv  = out["profile"].apply(lambda pf: band_center_and_span(pf, "rv")[0]).astype(float)
    centers_rho = out["profile"].apply(lambda pf: band_center_and_span(pf, "rho")[0]).astype(float)

    spans_dx  = out["profile"].apply(lambda pf: band_center_and_span(pf, "dx")[1]).astype(float)
    spans_rv  = out["profile"].apply(lambda pf: band_center_and_span(pf, "rv")[1]).astype(float)
    spans_rho = out["profile"].apply(lambda pf: band_center_and_span(pf, "rho")[1]).astype(float)

    def _emit(key, col_vals, c_vals, s_vals):
        dist = (col_vals.astype(float) - c_vals).abs()
        norm = (dist / s_vals).clip(0.0, 1.0)      # 0=center, 1=band edge
        out[f"{key}_to_center"]       = dist.round(3)
        out[f"{key}_to_center_norm"]  = norm.round(3)
        out[f"{key}_direction"]       = out[f"{key}_direction"] = np.where(col_vals < c_vals, "increase", "decrease")
        out[f"{key}_inside_band"]     = (col_vals >= (c_vals - s_vals)) & (col_vals <= (c_vals + s_vals))

    _emit("dx",  out[dxcol],  centers_dx,  spans_dx)
    _emit("rv",  out[rvcol],  centers_rv,  spans_rv)
    _emit("rho", out[rhocol], centers_rho, spans_rho)

    # Which lever is farthest from its center once profile weights are considered
    def _priority(row):
        w = PROFILES[row["profile"]]["weights"]
        return {
            "dx":  row["dx_to_center_norm"]  * w.get("dx",  0.0),
            "rv":  row["rv_to_center_norm"]  * w.get("rv",  0.0),
            "rho": row["rho_to_center_norm"] * w.get("rho", 0.0),
        }

    pri = out.apply(_priority, axis=1)
    out["priority_knob"]  = pri.apply(lambda d: max(d, key=d.get))
    out["priority_score"] = pri.apply(lambda d: max(d.values())).round(3)
    return out

def polish_df(df_in: pd.DataFrame) -> pd.DataFrame:
    """Optional: order columns for readability (keeps only those that exist)."""
    cols = [
        "delta_x","void_ratio","rupture_rho","K_lite","LSI_lite_100",
        "band_delta_x","band_r_v","band_rho_r","accepted","frame","profile",
        "dx_inside_band","dx_to_center","dx_to_center_norm","dx_direction",
        "rv_inside_band","rv_to_center","rv_to_center_norm","rv_direction",
        "rho_inside_band","rho_to_center","rho_to_center_norm","rho_direction",
        "priority_knob","priority_score",
    ]
    return df_in[[c for c in cols if c in df_in.columns]]

# ---- add the band-distance diagnostics & small polish ----
df = polish_df(add_band_distance_columns(df))
df

In [ ]:

# @title Plots
import matplotlib.pyplot as plt
if 'df' in globals() and not df.empty:
    xs = list(range(1, len(df)+1))
    plt.figure(figsize=(6,4))
    plt.plot(xs, df["delta_x"], label="Δx")
    plt.plot(xs, df["void_ratio"], label="r_v")
    plt.plot(xs, df["rupture_rho"], label="ρ_r")
    plt.xlabel("iteration"); plt.ylabel("value (0–1)"); plt.title("Primitives over iterations")
    plt.legend(); plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(xs, df["LSI_lite_100"])
    plt.xlabel("iteration"); plt.ylabel("LSI_lite_100 (0–100)"); plt.title("LSI_lite_100 over iterations")
    plt.show()
else:
    print("No dataframe to plot. Run scoring first.")


In [ ]:

# @title Export (CSV + simple HTML)
import pandas as pd, os
CSV_PATH = "/content/LSI_lite_results.csv"
HTML_PATH = "/content/LSI_lite_report.html"
if 'df' in globals() and not df.empty:
    df.to_csv(CSV_PATH, index=False)
    html = "<h2>LSI_lite results</h2>" + df.to_html(index=False)
    with open(HTML_PATH, "w") as f:
        f.write(html)
    print("Saved:", CSV_PATH, "and", HTML_PATH)
else:
    print("Nothing to export; run scoring first.")


### Notes & tips
- If the classic uploader throws a browser **RangeError** on big files, use **ZIP** or **Drive**.
- Keep `PIN_DEPS=False` unless you *need* exact versions; pinning can be slow in Colab.
- `Auto` is for smoke tests. For grading, set a fixed profile (esp. MarkMaking vs Landscape).
- If you see false **RED ρᵣ** on charcoal: widen the `rho.guard` upper bound OR increase `halo_frac`.
- Acceptance rule: `LSI_lite_100 ≥ gate` **and** all bands `OK`.
